# STRATUS eye-tracking case study: participant-safe, metric-safe evaluation

This notebook evaluates two implementations of the same regime-aware preprocessing model:

- **STRATUS-D** uses interpretable diagnostic potentials and Viterbi decoding.
- **STRATUS-BW** learns a diagonal-Gaussian HMM with Baum--Welch and then uses Viterbi decoding.
- **Oracle policy** applies the shared state-to-action policy to the known injected regimes. It is an upper bound for the action policy, not a deployable method.

The experiment separates numerical reconstruction from local action quality. Local action quality asks whether a method reconstructs short gaps, leaves long losses missing, preserves genuine movement and stable samples, and reduces error in injected unstable regions.

The split is **participant-level**. Baum--Welch is fitted only on training participants. Its permutation-invariant components are mapped to operational regime names using training labels only. All reported method comparisons use held-out participants.

In [ ]:
from pathlib import Path
import sys
import shutil
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from stratus.loaders import (
    load_dataset,
    estimate_sampling_rate_hz,
    estimate_segment_sampling_rate_hz,
    extract_valid_segments,
    infer_coordinate_bounds,
    sampling_rate_report,
)
from stratus.diagnostics import compute_diagnostics
from stratus.corruptions import inject_corruption
from stratus.baselines import (
    raw_degraded,
    global_interpolate,
    global_smooth,
    interp_smooth,
    robust_clip,
    short_gap_only,
)
from stratus.regimes import (
    predict_regimes_stratus,
    apply_state_to_action,
    apply_oracle_policy,
)
from stratus.baum_welch import BaumWelchSTRATUS
from stratus.metrics import evaluate_output, regime_f1
from stratus.plotting import save_bar, save_example_plot, save_grouped_bar, save_tradeoff_scatter

RESULTS = PROJECT_ROOT / "results"
FIGURES = RESULTS / "figures"
TABLES = RESULTS / "tables"
PAPER_FIGURES = PROJECT_ROOT / "paper" / "figures"

for folder in [FIGURES, TABLES, PAPER_FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

## 1. Configuration

The notebook loads up to the first ten CSV exports from each local dataset folder. Because an autism export may contain several participants, reference extraction selects at most one clean window **per participant**, not per source file.

Old generated tables and figures are removed before a new run, so all files in `results/` and `paper/figures/` belong to the current v7 experiment.

Coordinate plausibility bounds are estimated reproducibly from finite raw coordinates before corruption, using the 99.9th percentile rounded upward to the nearest 100 coordinate units. They are empirical coordinate-domain bounds, not claims about physical monitor resolution.

In [ ]:
import os

DYSLEXIA_DIR = Path(
    os.environ.get("STRATUS_ETDD70_DIR", PROJECT_ROOT / "data" / "raw" / "etdd70")
)
AUTISM_DIR = Path(
    os.environ.get("STRATUS_AUTISM_DIR", PROJECT_ROOT / "data" / "raw" / "autism")
)

for dataset_name, dataset_dir in {
    "ETDD70": DYSLEXIA_DIR,
    "Autism": AUTISM_DIR,
}.items():
    if not dataset_dir.exists():
        raise FileNotFoundError(
            f"{dataset_name} folder not found: {dataset_dir}\n"
            "Place the data under data/raw/ as described in data/README.md, "
            "or set STRATUS_ETDD70_DIR and STRATUS_AUTISM_DIR."
        )

N_FILES_PER_DATASET = 10
SEGMENT_SECONDS = 8.0
MAX_SEGMENTS_PER_PARTICIPANT = 1
MIN_VALID_FRACTION = 0.98

TEST_FRACTION = 0.30
SPLIT_SEED = 42

# Main held-out stress test.
EVAL_SEEDS = list(range(10))
SEVERITIES = ["low", "medium", "high"]
CORRUPTIONS = ["short_gap", "long_gap", "jitter", "spike", "mixed"]

# Baum--Welch training is deliberately smaller than evaluation to keep runtime manageable.
BW_TRAIN_SEEDS = [0, 1, 2, 3, 4]
BW_MAX_TRAIN_SEQUENCES_PER_DATASET = 120
BW_ITERATIONS = 50

OVERWRITE_RESULTS = True

if OVERWRITE_RESULTS:
    for folder in [FIGURES, TABLES, PAPER_FIGURES]:
        for path in folder.iterdir():
            if path.is_file():
                path.unlink()


### Final v7 configuration

The participant-safe extraction introduced in v6 is retained. Version 7 changes only evaluation and baseline validity:

- `short_gap_only` now interpolates a missing run only when its **complete run length** is at most 0.5 seconds;
- stable preservation is measured explicitly;
- repair of injected jitter and spikes is measured relative to the degraded input;
- the former plausibility-heavy “operator correctness” value is replaced by a transparent five-component **local action score**;
- the primary paper table uses an equal-weight macro-average over datasets;
- participant-level hierarchical bootstrap intervals quantify uncertainty.

Baum--Welch settings remain five training corruption seeds, at most 120 sequences per dataset, 50 iterations, and K-Means with 30 restarts.

## 2. Load and harmonize the datasets

Both source formats are converted to canonical columns including
`dataset`, `source_file`, `participant_id`, `recording_id`, `time_s`, `x`, and `y`.

ETDD70 combines valid left and right gaze coordinates and is expected near 250 Hz. The autism exports contain multiple participants per CSV and use millisecond timestamps. Sampling rates and coordinate bounds are reported rather than assumed.

In [ ]:
dyslexia = load_dataset(DYSLEXIA_DIR, dataset="dyslexia", n_files=N_FILES_PER_DATASET)
autism = load_dataset(AUTISM_DIR, dataset="autism", n_files=N_FILES_PER_DATASET)

DATASETS_RAW = {"ETDD70": dyslexia, "Autism": autism}
FS_BY_DATASET = {
    name: estimate_sampling_rate_hz(df)
    for name, df in DATASETS_RAW.items()
}
BOUNDS_BY_DATASET = {
    name: infer_coordinate_bounds(df)
    for name, df in DATASETS_RAW.items()
}

metadata_rows = []
for name, df in DATASETS_RAW.items():
    width, height = BOUNDS_BY_DATASET[name]
    metadata_rows.append({
        "dataset": name,
        "rows": len(df),
        "files": df["source_file"].nunique(),
        "participants": df["participant_id"].nunique(),
        "sampling_rate_hz": FS_BY_DATASET[name],
        "coordinate_width": width,
        "coordinate_height": height,
    })
    report = sampling_rate_report(df)
    report.insert(0, "dataset", name)
    report.to_csv(TABLES / f"{name.lower()}_sampling_rate_report.csv", index=False)

dataset_metadata = pd.DataFrame(metadata_rows)
dataset_metadata.to_csv(TABLES / "dataset_metadata.csv", index=False)
dataset_metadata

**What to inspect:** Dataset-level rates and per-recording timing reports should be plausible. Gap thresholds are derived from each extracted segment's own timestamps, not from a single hard-coded sample count. Coordinate bounds are saved in `dataset_metadata.csv` and used consistently for diagnostics, clipping, and plausibility reporting.

## 3. Extract exact-duration reference segments and create a participant-level split

A candidate window is built within one participant recording and must span at
least eight seconds without invalid coordinates or a large timestamp
interruption. One candidate per participant is selected reproducibly with seed
42. Participants, rather than source files, are then assigned to train or test.
This prevents the same person from occurring in both partitions when several
participants are stored in one autism CSV export.

In [ ]:
segments = []
extraction_rows = []

for name, df in DATASETS_RAW.items():
    seg = extract_valid_segments(
        df,
        segment_seconds=SEGMENT_SECONDS,
        max_segments_per_participant=MAX_SEGMENTS_PER_PARTICIPANT,
        min_valid_fraction=MIN_VALID_FRACTION,
        selection_seed=SPLIT_SEED,
    )
    segments.append(seg)
    extraction_rows.append({
        "dataset": name,
        "loaded_files": df["source_file"].nunique(),
        "available_participants": df["participant_id"].nunique(),
        "usable_participants": seg["participant_id"].nunique(),
        "reference_segments": seg["case_segment"].nunique(),
        "median_segment_seconds": seg.groupby("case_segment")["time_s"].agg(lambda s: s.max()-s.min()).median(),
    })

clean_segments = pd.concat(segments, ignore_index=True)
extraction_summary = pd.DataFrame(extraction_rows)
extraction_summary.to_csv(TABLES / "reference_extraction_summary.csv", index=False)
clean_segments.to_csv(TABLES / "clean_reference_segments.csv", index=False)

split_rows = []
train_parts, test_parts = [], []

for dataset, group in clean_segments.groupby("dataset", sort=False):
    participants = np.array(sorted(group["participant_id"].astype(str).unique()))
    local_rng = np.random.default_rng(SPLIT_SEED)
    local_rng.shuffle(participants)
    n_test = max(1, int(round(len(participants) * TEST_FRACTION)))
    if len(participants) > 1:
        n_test = min(n_test, len(participants) - 1)
    test_ids = set(participants[:n_test])
    train_ids = set(participants[n_test:])

    train = group[group["participant_id"].astype(str).isin(train_ids)].copy()
    test = group[group["participant_id"].astype(str).isin(test_ids)].copy()
    train_parts.append(train)
    test_parts.append(test)

    for participant in sorted(train_ids):
        split_rows.append({"dataset": dataset, "participant_id": participant, "split": "train"})
    for participant in sorted(test_ids):
        split_rows.append({"dataset": dataset, "participant_id": participant, "split": "test"})

train_segments = pd.concat(train_parts, ignore_index=True)
test_segments = pd.concat(test_parts, ignore_index=True)
split_table = pd.DataFrame(split_rows)
split_table.to_csv(TABLES / "participant_split.csv", index=False)

display(extraction_summary)
display(split_table.groupby(["dataset", "split"]).size().rename("participants").reset_index())

## 4. Train the Baum--Welch HMM variant

For every training reference segment, we inject controlled corruptions and compute the same six diagnostic variables used by STRATUS-D. A diagonal-Gaussian HMM with five components is then fitted by Baum--Welch.

Because HMM component numbers have no inherent semantics, a component learned as `0` is not automatically `Stable`. We solve this label-switching problem with a one-to-one assignment between components and operational regime names based on **training data only**. This mapping is necessary because the downstream state-to-action policy requires interpretable states.

STRATUS-BW therefore tests whether learned emissions and transitions improve the regime model. It does not receive test labels.

In [ ]:
bw_models = {}
bw_training_summary = []

for dataset, clean_group in train_segments.groupby("dataset", sort=False):
    width, height = BOUNDS_BY_DATASET[dataset]

    candidates = []
    for case_id, clean in clean_group.groupby("case_segment", sort=False):
        clean = clean.sort_values("time_s").reset_index(drop=True)
        for seed in BW_TRAIN_SEEDS:
            for severity in SEVERITIES:
                for corruption in CORRUPTIONS:
                    degraded = inject_corruption(
                        clean,
                        corruption=corruption,
                        severity=severity,
                        seed=seed,
                    )
                    diag = compute_diagnostics(
                        degraded,
                        screen_width=width,
                        screen_height=height,
                    )
                    candidates.append((diag, degraded["true_regime"].copy()))

    # Fixed random subset for reproducible and bounded training time.
    local_rng = np.random.default_rng(SPLIT_SEED)
    if len(candidates) > BW_MAX_TRAIN_SEQUENCES_PER_DATASET:
        chosen = local_rng.choice(
            len(candidates),
            size=BW_MAX_TRAIN_SEQUENCES_PER_DATASET,
            replace=False,
        )
        candidates = [candidates[i] for i in chosen]

    diagnostic_frames = [item[0] for item in candidates]
    true_sequences = [item[1] for item in candidates]

    model = BaumWelchSTRATUS.fit(
        diagnostic_frames,
        true_sequences,
        n_iter=BW_ITERATIONS,
        random_state=SPLIT_SEED,
    )
    bw_models[dataset] = model

    bw_training_summary.append({
        "dataset": dataset,
        "training_participants": clean_group["participant_id"].nunique(),
        "training_sequences": len(candidates),
        "baum_welch_iterations": len(model.hmm.log_likelihood_history_),
        "final_log_likelihood": model.hmm.log_likelihood_history_[-1],
        "component_mapping": str(model.component_to_state),
    })

bw_training_summary = pd.DataFrame(bw_training_summary)
bw_training_summary.to_csv(TABLES / "baum_welch_training_summary.csv", index=False)
bw_training_summary

**What to inspect:** The final log likelihood should be finite, and each of the five components should receive one semantic state label. A learned model may still perform worse than the diagnostic model; that would indicate that unconstrained statistical clusters do not align perfectly with preprocessing semantics.

## 5. Define comparison methods

All adaptive variants use the same policy:

- `Stable` and `Movement`: preserve;
- `LossShort`: interpolate;
- `LossLong`: keep missing;
- `Unstable`: centered rolling-median correction.

The three adaptive variants differ only in the source of the regime path: diagnostic potentials (STRATUS-D), Baum--Welch learning (STRATUS-BW), or injected ground truth (Oracle policy).

The crucial baseline correction in v7 is `short_gap_only`. It first determines every complete contiguous missing run and interpolates the run only when its total length does not exceed 0.5 seconds. It no longer uses Pandas' `limit=` behavior, which can partly fill both ends of a long gap.

| Method | Implementation | Local regime awareness |
|---|---|---|
| `raw_degraded` | Return the corrupted stream unchanged. | No |
| `global_interpolate` | Linear interpolation across all gaps. | No |
| `global_smooth` | Centered rolling mean, window 7. | No |
| `interp_smooth` | Global interpolation followed by rolling mean. | No |
| `robust_clip` | Clip finite coordinates to empirical bounds. | No |
| `short_gap_only` | Run-based interpolation only for complete gaps no longer than 0.5 seconds. | Gap-length rule only |
| `STRATUS-D` | Six diagnostics, interpretable potentials, persistent Viterbi path, shared action policy. | Yes |
| `STRATUS-BW` | Five-state diagonal-Gaussian HMM; log/robust transform; K-Means `n_init=30`; up to 50 Baum--Welch iterations; training-only Hungarian semantic alignment; shared action policy. | Yes |
| `Oracle_policy` | Injected regime path with the shared policy. | Perfect reference path |

In [ ]:
METHODS = [
    "raw_degraded",
    "global_interpolate",
    "global_smooth",
    "interp_smooth",
    "robust_clip",
    "short_gap_only",
    "STRATUS-D",
    "STRATUS-BW",
    "Oracle_policy",
]


def run_method(method_name, degraded, dataset):
    fs_hz = estimate_segment_sampling_rate_hz(degraded)
    if not np.isfinite(fs_hz):
        raise ValueError("Could not estimate segment sampling rate")
    width, height = BOUNDS_BY_DATASET[dataset]
    max_gap_samples = max(1, int(round(0.50 * fs_hz)))

    if method_name == "raw_degraded":
        return raw_degraded(degraded)
    if method_name == "global_interpolate":
        return global_interpolate(degraded)
    if method_name == "global_smooth":
        return global_smooth(degraded)
    if method_name == "interp_smooth":
        return interp_smooth(degraded)
    if method_name == "robust_clip":
        return robust_clip(degraded, screen_width=width, screen_height=height)
    if method_name == "short_gap_only":
        return short_gap_only(degraded, max_gap_samples=max_gap_samples)

    diag = compute_diagnostics(degraded, screen_width=width, screen_height=height)

    if method_name == "STRATUS-D":
        regimes = predict_regimes_stratus(diag, fs_hz=fs_hz, short_gap_seconds=0.50)
        return apply_state_to_action(diag, regimes, stable_smoothing=False)

    if method_name == "STRATUS-BW":
        regimes = bw_models[dataset].predict(diag)
        return apply_state_to_action(diag, regimes, stable_smoothing=False)

    if method_name == "Oracle_policy":
        return apply_oracle_policy(diag)

    raise ValueError(method_name)

## 6. Held-out stress-test experiment

Every method is evaluated on the same corrupted streams from held-out participants. The design crosses participant reference windows, ten random seeds, three severity levels, and five corruption settings.

The injected stream retains `x_degraded` and `y_degraded` as immutable columns. This enables a direct unstable-region repair metric even after a method overwrites the working `x` and `y` columns.

In [ ]:
rows = []
f1_rows = []
example_saved = set()

for case_id, clean in test_segments.groupby("case_segment", sort=False):
    clean = clean.sort_values("time_s").reset_index(drop=True)
    dataset = clean["dataset"].iloc[0]
    width, height = BOUNDS_BY_DATASET[dataset]

    for seed in EVAL_SEEDS:
        for severity in SEVERITIES:
            for corruption in CORRUPTIONS:
                degraded = inject_corruption(
                    clean,
                    corruption=corruption,
                    severity=severity,
                    seed=seed,
                )

                for method in METHODS:
                    output = run_method(method, degraded.copy(), dataset=dataset)
                    metrics = evaluate_output(
                        output,
                        screen_width=width,
                        screen_height=height,
                    )
                    rows.append({
                        "dataset": dataset,
                        "case_segment": case_id,
                        "source_file": clean["source_file"].iloc[0],
                        "participant_id": clean["participant_id"].iloc[0],
                        "seed": seed,
                        "severity": severity,
                        "corruption": corruption,
                        "method": method,
                        **metrics,
                    })

                    if method in {"STRATUS-D", "STRATUS-BW", "Oracle_policy"}:
                        f1 = regime_f1(output)
                        for _, r in f1.iterrows():
                            f1_rows.append({
                                "dataset": dataset,
                                "case_segment": case_id,
                                "seed": seed,
                                "severity": severity,
                                "corruption": corruption,
                                "method": method,
                                "regime": r["regime"],
                                "f1": r["f1"],
                                "support": r["support"],
                            })

                    key = (dataset, method)
                    if (
                        key not in example_saved
                        and corruption == "mixed"
                        and severity == "medium"
                        and seed == 0
                        and method in {"STRATUS-D", "STRATUS-BW"}
                    ):
                        safe_method = method.lower().replace("-", "_")
                        save_example_plot(
                            output,
                            FIGURES / f"example_{safe_method}_{dataset.lower()}.pdf",
                            title=f"{method}: {dataset}, mixed corruption",
                        )
                        example_saved.add(key)

results = pd.DataFrame(rows)
f1_results = pd.DataFrame(f1_rows)
results.to_csv(TABLES / "stress_test_results.csv", index=False)
f1_results.to_csv(TABLES / "regime_f1_results.csv", index=False)

print("Held-out result rows:", len(results))
print("Test participants:", test_segments["participant_id"].nunique())
results.head()

## 7. Aggregate paper tables

The primary table is an equal-weight macro-average over the two datasets. A separate pooled table is retained for transparency but is not used as the headline result because the held-out Autism group is larger than ETDD70.

The **local action score** is the unweighted mean of five separately reported components after aggregation:

1. long-gap conservatism (`1 - long_gap_hallucination`),
2. short-gap recovery,
3. movement preservation,
4. exact stable-sample preservation,
5. clipped unstable-region repair gain.

It is a paper-specific summary, not a universal data-quality metric. The individual components remain the primary evidence. Unstable repair gain itself is also saved without clipping, so harmful processing remains visible.

In [ ]:
COMPONENT_COLUMNS = [
    "long_gap_conservatism",
    "short_recovery",
    "movement_preservation",
    "stable_preservation",
    "unstable_repair_score",
]
METRIC_COLUMNS = [
    "mae", "rmse", "retention", "long_hallucination", "short_recovery",
    "movement_preservation", "stable_preservation", "unstable_repair_gain",
    "unstable_repair_score", "plausibility_score",
]


def aggregate_metrics(frame, group_columns):
    table = frame.groupby(group_columns, as_index=False).agg(
        **{column: (column, "mean") for column in METRIC_COLUMNS}
    )
    table["long_gap_conservatism"] = 1.0 - table["long_hallucination"]
    table["local_action_score"] = table[COMPONENT_COLUMNS].mean(axis=1, skipna=False)
    return table


# Pooled across all held-out cases; Autism receives more weight because it has more participants.
pooled_summary = aggregate_metrics(results, ["method"]).sort_values(
    "local_action_score", ascending=False
)
pooled_summary.to_csv(TABLES / "pooled_results.csv", index=False)
pooled_summary.to_latex(TABLES / "pooled_results.tex", index=False, float_format="%.3f")

# First aggregate within each dataset, then average the two datasets equally.
dataset_summary = aggregate_metrics(results, ["dataset", "method"]).sort_values(
    ["dataset", "local_action_score"], ascending=[True, False]
)
macro_summary = (
    dataset_summary.groupby("method", as_index=False)
    .agg(**{column: (column, "mean") for column in METRIC_COLUMNS + ["long_gap_conservatism"]})
)
macro_summary["local_action_score"] = macro_summary[COMPONENT_COLUMNS].mean(axis=1, skipna=False)
macro_summary = macro_summary.sort_values("local_action_score", ascending=False)

dataset_summary.to_csv(TABLES / "dataset_results.csv", index=False)
dataset_summary.to_latex(TABLES / "dataset_results.tex", index=False, float_format="%.3f")
macro_summary.to_csv(TABLES / "macro_results.csv", index=False)
macro_summary.to_latex(TABLES / "macro_results.tex", index=False, float_format="%.3f")

macro_summary

In [ ]:
# Participant-level estimates and hierarchical bootstrap intervals.
participant_summary = aggregate_metrics(
    results,
    ["dataset", "participant_id", "method"],
)
participant_summary.to_csv(TABLES / "participant_results.csv", index=False)


def hierarchical_bootstrap_ci(participant_table, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    bootstrap_rows = []
    methods = participant_table["method"].unique()
    datasets = participant_table["dataset"].unique()
    value_columns = METRIC_COLUMNS + ["long_gap_conservatism"]

    for method in methods:
        method_data = participant_table[participant_table["method"] == method]
        draws = {column: [] for column in value_columns + ["local_action_score"]}

        for _ in range(n_boot):
            dataset_means = []
            for dataset in datasets:
                part = method_data[method_data["dataset"] == dataset]
                if part.empty:
                    continue
                sampled_positions = rng.integers(0, len(part), size=len(part))
                sampled = part.iloc[sampled_positions]
                dataset_means.append(sampled[value_columns].mean())

            macro = pd.DataFrame(dataset_means).mean()
            for column in value_columns:
                draws[column].append(float(macro[column]))
            draws["local_action_score"].append(
                float(macro[COMPONENT_COLUMNS].mean(skipna=False))
            )

        row = {"method": method, "bootstrap_replicates": n_boot}
        for column, values in draws.items():
            row[f"{column}_ci_low"] = float(np.nanquantile(values, 0.025))
            row[f"{column}_ci_high"] = float(np.nanquantile(values, 0.975))
        bootstrap_rows.append(row)
    return pd.DataFrame(bootstrap_rows)


bootstrap_ci = hierarchical_bootstrap_ci(participant_summary)
bootstrap_ci.to_csv(TABLES / "participant_bootstrap_ci.csv", index=False)
macro_summary_with_ci = macro_summary.merge(bootstrap_ci, on="method", how="left")
macro_summary_with_ci.to_csv(TABLES / "macro_results_with_ci.csv", index=False)
macro_summary_with_ci.to_latex(
    TABLES / "macro_results_with_ci.tex", index=False, float_format="%.3f"
)
macro_summary_with_ci[[
    "method", "mae", "mae_ci_low", "mae_ci_high",
    "local_action_score", "local_action_score_ci_low", "local_action_score_ci_high",
]]

In [ ]:
corruption_summary = aggregate_metrics(results, ["corruption", "method"])
severity_summary = aggregate_metrics(results, ["severity", "method"])

corruption_summary.to_csv(TABLES / "corruption_results.csv", index=False)
corruption_summary.to_latex(TABLES / "corruption_results.tex", index=False, float_format="%.3f")
severity_summary.to_csv(TABLES / "severity_results.csv", index=False)
severity_summary.to_latex(TABLES / "severity_results.tex", index=False, float_format="%.3f")

display(corruption_summary.head())
display(severity_summary.head())

In [ ]:
regime_f1_summary = (
    f1_results
    .groupby(["method", "regime"], as_index=False)
    .agg(
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),
        evaluated_cases=("f1", "count"),
        total_support=("support", "sum"),
    )
    .sort_values(["method", "mean_f1"], ascending=[True, False])
)
regime_f1_summary.to_csv(TABLES / "regime_f1_summary.csv", index=False)
regime_f1_summary.to_latex(TABLES / "regime_f1_summary.tex", index=False, float_format="%.3f")
regime_f1_summary

**What to inspect:** Compare both learned regime paths with the oracle, but do not infer action quality from F1 alone. The corrected short-gap baseline is expected to be strong on gap handling and stable preservation while scoring near zero on unstable repair. STRATUS must earn any advantage through the combination of regime-specific capabilities, not through a defective baseline.

## 8. Regenerate paper figures from dataset-macro results

In [ ]:
save_bar(
    macro_summary.sort_values("mae"),
    x="method", y="mae",
    title="Dataset-macro reconstruction error by method",
    path=FIGURES / "mae_by_method.pdf", ylabel="MAE",
)
save_bar(
    macro_summary.sort_values("local_action_score", ascending=False),
    x="method", y="local_action_score",
    title="Dataset-macro local action score by method",
    path=FIGURES / "local_action_score_by_method.pdf", ylabel="Local action score",
)
save_bar(
    macro_summary.sort_values("unstable_repair_gain", ascending=False),
    x="method", y="unstable_repair_gain",
    title="Repair gain in injected unstable regions",
    path=FIGURES / "unstable_repair_gain_by_method.pdf", ylabel="Unstable repair gain",
)
save_bar(
    macro_summary.sort_values("long_hallucination"),
    x="method", y="long_hallucination",
    title="Dataset-macro long-gap hallucination",
    path=FIGURES / "long_gap_hallucination_by_method.pdf", ylabel="Long-gap hallucination",
)
save_grouped_bar(
    regime_f1_summary,
    index="regime", columns="method", values="mean_f1",
    title="Regime inference F1 by variant",
    path=FIGURES / "regime_f1_by_variant.pdf", ylabel="Mean F1",
)
save_tradeoff_scatter(
    macro_summary,
    x="mae", y="local_action_score", label="method",
    title="Reconstruction error versus local action quality",
    path=FIGURES / "mae_action_tradeoff.pdf",
    xlabel="Dataset-macro MAE", ylabel="Local action score",
)

for pdf in FIGURES.glob("*.pdf"):
    shutil.copy2(pdf, PAPER_FIGURES / pdf.name)

print("Figures regenerated in:", FIGURES)
print("Current paper copies in:", PAPER_FIGURES)

## 9. Automatic validity checks and interpretation

These checks protect the publication claims rather than enforcing a favorable result:

- no participant may occur in both train and test;
- every reference window must span at least eight seconds;
- Baum--Welch likelihoods must be finite;
- the corrected short-gap baseline must never fill an injected long gap;
- the raw stream must have approximately zero unstable repair gain;
- the oracle should provide a strong reference for the shared policy;
- primary results must contain participant-level uncertainty intervals.

In [ ]:
assert dataset_metadata["sampling_rate_hz"].map(np.isfinite).all()
assert not set(train_segments["participant_id"]).intersection(set(test_segments["participant_id"]))
assert clean_segments.groupby("case_segment").apply(
    lambda g: g["time_s"].max() - g["time_s"].min(), include_groups=False
).ge(SEGMENT_SECONDS).all()
assert (dataset_metadata["coordinate_width"] > 0).all()
assert (dataset_metadata["coordinate_height"] > 0).all()
assert bw_training_summary["final_log_likelihood"].map(np.isfinite).all()
assert len(bw_models) == dataset_metadata["dataset"].nunique()

short_baseline = results[results["method"] == "short_gap_only"]
assert np.nanmax(short_baseline["long_hallucination"].to_numpy()) < 1e-12

raw_unstable = macro_summary.loc[
    macro_summary["method"] == "raw_degraded", "unstable_repair_gain"
].iloc[0]
assert abs(raw_unstable) < 1e-9
assert bootstrap_ci["bootstrap_replicates"].min() >= 1000

adaptive = macro_summary[
    macro_summary["method"].isin(["STRATUS-D", "STRATUS-BW", "Oracle_policy"])
].copy()
display(adaptive)

for method in ["short_gap_only", "STRATUS-D", "STRATUS-BW", "Oracle_policy"]:
    row = macro_summary[macro_summary["method"] == method].iloc[0]
    print(
        method,
        "| MAE:", round(row["mae"], 3),
        "| long hallucination:", round(row["long_hallucination"], 3),
        "| short recovery:", round(row["short_recovery"], 3),
        "| stable preservation:", round(row["stable_preservation"], 3),
        "| unstable repair gain:", round(row["unstable_repair_gain"], 3),
        "| local action score:", round(row["local_action_score"], 3),
    )

print("\nRun complete. Use macro_results_with_ci.csv as the primary paper table; pooled_results.csv is a secondary sensitivity view.")